# QuetHoaDon · Web demo quét hóa đơn (Gradio trên Kaggle GPU)

Mở một trang web: tải ảnh hóa đơn lên → hệ thống trả về **người bán, địa chỉ, ngày giờ, tổng tiền** và **từng mặt hàng**,
kèm ảnh có khung chữ tô màu theo trường. Model là **cấu hình cuối đã đóng băng** (giống notebook 07): tiền xử lý → CRAFT
(lọc hộp chồng nhau 0,5) → VietOCR tinh chỉnh → M1b v3 → luật ngày giờ, làm sạch SELLER / ADDRESS → đọc mặt hàng.

**Cách chạy (khác notebook 07):** cần phiên **tương tác**, vì trang web chỉ sống khi notebook còn chạy.
1. Settings: **GPU T4**, **Internet On**. Gắn 2 dataset: **Vietnamese Receipts MC_OCR 2021** và `phase1-result`.
2. Bấm **Run All** ngay trong trình soạn thảo (không dùng Save & Run All).
3. Cell 5 in ra một link `https://….gradio.live`: mở link, ai có link cũng dùng được (tới 72 giờ, khi phiên còn chạy).
4. Giữ tab Kaggle mở: Kaggle tắt phiên khi để yên quá lâu. Dùng xong thì chạy cell 6 hoặc **Stop session** để không tốn giờ GPU.

Thời gian: khoảng 10–15 phút để cài, sau đó mỗi hóa đơn mất khoảng 3 giây.

In [ ]:
# CELL 1 - Cài detectron2 (LayoutXLM cần, 5-10 phút). Mong đợi: "detectron2 OK"
try:
    import detectron2
    print("detectron2 đã có:", detectron2.__version__)
except ImportError:
    !pip install -q 'git+https://github.com/facebookresearch/detectron2.git' --no-build-isolation 2>&1 | tail -3
    import detectron2; print("detectron2 OK", detectron2.__version__)

In [ ]:
# CELL 2 - Lấy mã nguồn nhánh main, cài thư viện OCR và Gradio. Mong đợi: một dòng commit, "gradio ...", "xong"
BRANCH = "main"
import os
%cd /kaggle/working
if not os.path.exists('QuetHoaDon'):
    !git clone -q https://github.com/CaoLePhucThinh2203/QuetHoaDon.git
%cd /kaggle/working/QuetHoaDon
!git fetch -q origin && git checkout -q -f -B {BRANCH} origin/{BRANCH} && git log --oneline -1
!pip install -q -r requirements-dev.txt
!pip install -q craft-text-detector vietocr --no-deps
!pip install -q gdown einops prefetch_generator albumentations lmdb gradio 2>&1 | tail -2
import gradio; print("gradio", gradio.__version__)
print("xong")

In [ ]:
# CELL 3 [KIỂM THỬ] - GPU, ảnh gốc, checkpoint M1b v3, trọng số VietOCR tinh chỉnh và unit test.
# Mong đợi: "GPU: True Tesla T4", mọi đường dẫn có giá trị, "Ran ... tests" rồi "OK"
import glob, os, sys, json, torch
REPO = '/kaggle/working/QuetHoaDon'
sys.path.insert(0, REPO); sys.path.insert(0, f'{REPO}/scripts')
print("GPU:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
one = lambda pattern: sorted(glob.glob(pattern, recursive=True))[-1]
IMG_RAW = os.path.dirname(one('/kaggle/input/**/train_images/**/*.jpg'))
TOKC = os.path.dirname(one('/kaggle/input/**/phase1_eval/tok_ft_n05/test.jsonl'))
CK3 = os.path.dirname(one('/kaggle/input/**/m1b_v3_checkpoint/best/config.json'))
VFT1 = one('/kaggle/input/**/phase1_eval/vietocr_ft/vietocr_finetuned.pth')
for k in ("IMG_RAW", "TOKC", "CK3", "VFT1"):
    print(f"{k} = {globals()[k]}")
!cd {REPO} && python -m unittest discover -s tests 2>&1 | grep -E "^(Ran|OK|FAILED|ERROR|FAIL)"

In [ ]:
# CELL 4 [KIỂM THỬ] - Nạp cấu hình cuối một lần, rồi kiểm tra đường đi của web trên 2 hóa đơn test (GPU, 1-2 phút).
# Mong đợi: hai dòng "web = đường đi đã chấm: KHỚP" và hai dòng "so với notebook 07: KHỚP"
from src.pipeline.extractor import ReceiptExtractor, compact_result
from src.training.data import load_jsonl
import demo_app

ex = ReceiptExtractor.from_checkpoint(CK3, f'{REPO}/configs', 'cuda', with_ocr=True, postprocess=True,
                                      ocr_weights=VFT1, items=True, nested_threshold=0.5)
scan = demo_app.make_scan(ex)

# Kết quả của cell 10 notebook 07 (cùng cấu hình, cùng ảnh): dùng để kiểm tra web cho ra đúng như vậy
NB07 = {
    "mcocr_public_145013dhzmu.jpg": {"fields": {"TOTAL_COST": {"text": "418,986", "amount_vnd": 418986}}, "n_items": 0},
    "mcocr_public_145013tumhj.jpg": {"fields": {"TIMESTAMP": {"text": "15/08/2020", "date": "15/08/2020"},
                                                "TOTAL_COST": {"text": "585,500", "amount_vnd": 585500},
                                                "SELLER": {"text": "Phúc Anh Minimart"},
                                                "ADDRESS": {"text": "248 Cổ Bi - Gia Lâm Hà Nội"}}, "n_items": 4},
}
for name, expected in NB07.items():
    path = os.path.join(IMG_RAW, name)
    web = scan(path)[5]                                   # JSON the web page shows
    direct = compact_result(ex.extract(path))             # the path scored in notebook 07
    print(name, "| web = đường đi đã chấm:", "KHỚP" if web == direct else "LỆCH")
    got = {"fields": web["fields"], "n_items": len(web.get("items", []))}
    print(name, "| so với notebook 07:", "KHỚP" if got == expected else f"LỆCH, được {got}")

# Ảnh mẫu dưới ô tải lên: 2 ảnh trên + 4 ảnh test kế tiếp theo thứ tự trong file, không chọn lọc
test_images = list(dict.fromkeys(r['image'] for r in load_jsonl(os.path.join(TOKC, 'test.jsonl'))))
EXAMPLES = [os.path.join(IMG_RAW, n) for n in NB07] + [os.path.join(IMG_RAW, n) for n in test_images if n not in NB07][:4]
EXAMPLES = [p for p in EXAMPLES if os.path.exists(p)]
print(len(EXAMPLES), "ảnh mẫu")

In [ ]:
# CELL 5 - Mở web demo. Mong đợi: dòng "Running on public URL: https://....gradio.live". Mở link đó để dùng.
# Cell chạy xong ngay nhưng trang web vẫn sống cho tới khi chạy cell 6 hoặc tắt phiên.
demo = demo_app.build_ui(scan, EXAMPLES)
demo.queue(max_size=16).launch(share=True, server_name="0.0.0.0", server_port=7860, quiet=False)

In [ ]:
# CELL 6 - Tắt web demo khi dùng xong (rồi Stop session để không tốn giờ GPU)
demo.close(); print("đã tắt")